## 1. Hugging Face Login

In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 2. Check GPU

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## 4. Load Tokenizer

In [ ]:
from collections import defaultdict, Counter
import heapq


SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]


class BytelevelBPE:

    def __init__(self, vocab_size=5000):
        self.vocab_size = vocab_size
        self.specials = SPECIALS.copy()
        self.byte_vocab = [str(i) for i in range(256)]
        self.merges = []
        self.merge_ranks = {}
        self.vocab = []
        self.stoi = {}
        self.itos = {}
        # Maps each BPE symbol to the bytes represented by it
        self.symbol_to_bytes = {}
    
    # 1. WORD -> UTF-8 BYTE SYMBOLS

    def word_to_symbols(self, word):
        byte_seq = word.encode("utf-8")
        return [str(b) for b in byte_seq] + ["</w>"]

    # 2. BUILD WORD FREQUENCIES

    def build_word_freqs(self, corpus):
        word_counter = Counter()
        print("Building word frequencies...")
        for text in corpus:
            if not isinstance(text, str):
                continue

            word_counter.update(text.split())
        print(
            f"Unique whitespace-separated words: "
            f"{len(word_counter):,}"
        )
        # Convert each unique word to byte symbols only once
        word_freqs = {}

        for word, freq in word_counter.items():
            symbols = tuple(self.word_to_symbols(word))
            word_freqs[symbols] = (
                word_freqs.get(symbols, 0) + freq
            )

        return word_freqs


    # 3. PAIRS OCCURRING IN ONE WORD

    @staticmethod
    def get_pairs(symbols):
        return {
            (symbols[i], symbols[i + 1])
            for i in range(len(symbols) - 1)
        }

    # 4. COUNT PAIR OCCURRENCES IN ONE WORD

    @staticmethod
    def pair_counts_in_word(symbols):
        counts = Counter()
        for i in range(len(symbols) - 1):
            counts[
                (symbols[i], symbols[i + 1])
            ] += 1

        return counts

    # 5. MERGE A PAIR INSIDE ONE WORD

    @staticmethod
    def merge_symbols(symbols, pair, merged_symbol):
        new_symbols = []
        i = 0
        while i < len(symbols):
            if (
                i < len(symbols) - 1
                and symbols[i] == pair[0]
                and symbols[i + 1] == pair[1]
            ):
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1

        return tuple(new_symbols)

    # 6. TRAIN OPTIMIZED BPE

    def train(self, corpus):
        word_freqs = self.build_word_freqs(corpus)
        vocab = (
            self.specials
            + self.byte_vocab
            + ["</w>"]
        )
        seen = set(vocab)
        for b in range(256):
            self.symbol_to_bytes[str(b)] = bytes([b])
        self.symbol_to_bytes["</w>"] = b""

        words = []
        frequencies = []
        for symbols, freq in word_freqs.items():
            words.append(symbols)
            frequencies.append(freq)
        del word_freqs

        pair_counts = defaultdict(int)
        pair_to_words = defaultdict(set)

        print("Building initial pair statistics...")

        for word_id, symbols in enumerate(words):
            freq = frequencies[word_id]
            local_counts = self.pair_counts_in_word(symbols)
            for pair, count in local_counts.items():
                pair_counts[pair] += count * freq
                pair_to_words[pair].add(word_id)

        print(
            f"Initial unique pairs: "
            f"{len(pair_counts):,}"
        )

        # MAX HEAP
        #
        # Python has min-heap, so use negative frequency.
        #
        # We keep stale entries and validate them when popped.
        # This avoids rebuilding the heap after every merge.

        heap = []
        for pair, count in pair_counts.items():
            heapq.heappush(heap,(-count, pair))
        target_merges = max(0,self.vocab_size - len(vocab))
        print(f"Target merges: {target_merges:,}")
        while (len(vocab) < self.vocab_size and heap):
            while heap:
                neg_count, pair = heapq.heappop(heap)
                heap_count = -neg_count
                current_count = pair_counts.get(pair,0)
                # Ignore stale heap entries
                if (current_count == heap_count and current_count > 0):
                    break
            else:
                break

            max_freq = pair_counts.get(pair, 0)

            # Same stopping condition as before
            if max_freq <= 1:
                break

            # Create collision-free merged symbol
            merged_symbol = (f"<bpe_{len(self.merges)}>")

            # Byte representation of new symbol
            left_bytes = self.symbol_to_bytes[pair[0]]
            right_bytes = self.symbol_to_bytes[pair[1]]
            self.symbol_to_bytes[merged_symbol] = left_bytes + right_bytes

            self.merges.append(pair)
            self.merge_ranks[pair] = (len(self.merges) - 1,merged_symbol)
            if merged_symbol not in seen:
                seen.add(merged_symbol)
                vocab.append(merged_symbol)

            # ----------------------------------------------------
            # IMPORTANT OPTIMIZATION
            #
            # Only process words containing selected pair.
            # ----------------------------------------------------

            affected_word_ids = list(pair_to_words.get(pair, set()))
            changed_pairs = set()
            for word_id in affected_word_ids:
                old_symbols = words[word_id]
                freq = frequencies[word_id]
                # Pair may have disappeared because of an earlier
                # update involving this word
                if pair not in self.get_pairs(old_symbols):
                    continue
                
                # Statistics BEFORE merge
                old_counts = self.pair_counts_in_word(old_symbols)

                # Merge selected pair
                new_symbols = self.merge_symbols(old_symbols,pair,merged_symbol)
                if new_symbols == old_symbols:
                    continue

                # Statistics AFTER merge
                new_counts = self.pair_counts_in_word(new_symbols)

                # Save updated word
                words[word_id] = new_symbols

                all_pairs = (
                    set(old_counts)
                    | set(new_counts)
                )

                # Update only pairs affected by this word

                for p in all_pairs:

                    old_n = old_counts.get(p, 0)
                    new_n = new_counts.get(p, 0)
                    delta = (
                        new_n - old_n
                    ) * freq

                    if delta != 0:
                        pair_counts[p] += delta
                        changed_pairs.add(p)

                    # Update inverted index
                    if new_n > 0:
                        pair_to_words[p].add(word_id)
                    else:
                        pair_to_words[p].discard(word_id)

            # Selected pair should now be gone from affected words
            pair_to_words[pair].clear()

            # Push updated frequencies into heap
            for p in changed_pairs:
                count = pair_counts.get(p, 0)
                if count > 0:
                    heapq.heappush(heap,(-count, p))

        self.vocab = vocab
        self.vocab_size = len(vocab)
        self.stoi = {
            symbol: i
            for i, symbol in enumerate(vocab)
        }

        self.itos = {
            i: symbol
            for symbol, i in self.stoi.items()
        }

        print("\nTokenizer training complete.")

        print(
            "Final vocabulary size:",
            self.vocab_size
        )

        print(
            "Number of merges:",
            len(self.merges)
        )

    # 7. APPLY BPE TO ONE WORD

    def apply_bpe(self, word):
        symbols = self.word_to_symbols(word)
        while len(symbols) > 1:
            best_pair = None
            best_rank = float("inf")
            best_merged_symbol = None

            for i in range(len(symbols) - 1):
                pair = (symbols[i],symbols[i + 1])
                merge_info = self.merge_ranks.get(pair)
                if merge_info is not None:
                    rank, merged_symbol = merge_info
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = pair
                        best_merged_symbol = merged_symbol

            if best_pair is None:
                break

            symbols = list(
                self.merge_symbols(
                    tuple(symbols),
                    best_pair,
                    best_merged_symbol
                )
            )

        return symbols

    # 8. ENCODE

    def encode(self,text,add_bos=False,add_eos=False):
        ids = []
        if add_bos:
            ids.append(
                self.stoi["<s>"]
            )

        for word in text.split():
            symbols = self.apply_bpe(word)
            for symbol in symbols:
                ids.append(
                    self.stoi.get(
                        symbol,
                        self.stoi["<unk>"]
                    )
                )

        if add_eos:
            ids.append(
                self.stoi["</s>"]
            )

        return ids

    # 9. DECODE

    def decode(self, ids):

        words = []
        current_bytes = bytearray()

        for token_id in ids:
            symbol = self.itos.get(token_id)
            if symbol is None:
                continue
            if symbol in self.specials:
                continue

            # Determine whether this token ends a word
            ends_word = self._symbol_ends_word(symbol)
            byte_value = self.symbol_to_bytes.get(symbol, b"")
            current_bytes.extend(byte_value)

            if ends_word:
                try:
                    word = bytes(current_bytes).decode("utf-8")
                    words.append(word)
                except UnicodeDecodeError:
                    words.append("<unk>")
                current_bytes = bytearray()

        # Remaining bytes
        if current_bytes:
            try:
                words.append(bytes(current_bytes).decode("utf-8"))
            except UnicodeDecodeError:
                words.append("<unk>")

        return " ".join(words)

    # 10. CHECK WHETHER SYMBOL CONTAINS </w> AT END

    def _symbol_ends_word(self, symbol):
        if symbol == "</w>":
            return True
        # Find merge that created this symbol
        for pair, (_, merged) in self.merge_ranks.items():
            if merged == symbol:
                return self._symbol_ends_word(
                    pair[1]
                )

        return False

    # 11. VOCAB SIZE

    def get_vocab_size(self):
        return len(self.vocab)

## 9. Define The Model

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        if self.d_head % 2 != 0:
            raise ValueError("RoPE requires an even head dimension.")

        inv_freq = 1.0 / (
            10000.0 ** (torch.arange(0, self.d_head, 2).float() / self.d_head)
        )
        angles = torch.outer(torch.arange(max_seq_len).float(), inv_freq)
        self.register_buffer("rope_cos", angles.cos(), persistent=False)
        self.register_buffer("rope_sin", angles.sin(), persistent=False)
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

    def apply_rope(self, q, k, offset=0):
        T = q.size(2)
        cos = self.rope_cos[offset:offset + T][None, None].to(q.dtype)
        sin = self.rope_sin[offset:offset + T][None, None].to(q.dtype)

        def rotate(x):
            first, second = x.chunk(2, dim=-1)
            return torch.cat(
                [first * cos - second * sin, first * sin + second * cos],
                dim=-1,
            )

        return rotate(q), rotate(k)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        q, k = self.apply_rope(q, k)

        scores = q @ k.transpose(-2, -1) * self.attention_scale

        scores = scores.masked_fill(self.causal_mask[:T, :T] == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        weighted_sum = attn_weights @ v
        weighted_sum = weighted_sum.transpose(1, 2).contiguous().view(B, T, C)

        out = self.out_proj(weighted_sum)
        out = self.resid_dropout(out)

        return out

In [ ]:
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 512
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4          
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "rope"   
    attention_scale: float = None      
    norm_first: bool = False            # norm_first=True is for pre-norm else post-norm


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        # TODO: two linear layers with a GELU in between, plus dropout
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout,
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

        self.norm_first = cfg.norm_first

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))
        else:
            x = self.ln1(x + self.attn(x))
            x = self.ln2(x + self.ffn(x))
        return x


class DecoderLM(nn.Module):
    """The model class the autograder imports directly. Do not rename it."""

    def __init__(self, cfg: LMConfig):
        super().__init__()
        if cfg.pos_emb_type != "rope":
            raise ValueError("This notebook requires a RoPE model/checkpoint.")
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T) LongTensor of token ids -> logits (B, T, vocab_size)."""
        B, T = idx.shape

        x = self.tok_emb(idx)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.head(x)
        return logits


# **Phase 4**

In [6]:
import json
import random
import zipfile

from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

from huggingface_hub import hf_hub_download

from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint

from torch.nn.utils.rnn import (
    pad_sequence,
    pack_padded_sequence,
    pad_packed_sequence,
)

from sklearn.metrics import (
    f1_score,
    classification_report,
)


MODEL_DIR = Path(
    "/kaggle/input/datasets/tejasvaidya137/model1-for-phase4-files"
)

OUTPUT_DIR = Path(
    "/kaggle/working/phase4_model1_bigru_attention"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42

CHUNK_LENGTH = 512
MAX_CHUNKS = 16

DOC_BATCH_SIZE = 4
CHUNK_BATCH_SIZE = 32

UNFREEZE_LAST_N = 2
USE_ACTIVATION_CHECKPOINTING = True

GRU_HIDDEN = 256

BACKBONE_LR = 1e-4
HEAD_LR = 5e-4

EPOCHS = 6
PATIENCE = 3

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

use_amp = device.type == "cuda"

amp_dtype = (
    torch.bfloat16
    if use_amp and torch.cuda.is_bf16_supported()
    else torch.float16
)

print("Device:", device)

Device: cuda


In [7]:
# Load pretrained model
saved = torch.load(
    MODEL_DIR / "model.pt",
    map_location="cpu",
    weights_only=True,
)

cfg = LMConfig(**saved["model_config"])

phase1_model = DecoderLM(cfg)
phase1_model.load_state_dict(saved["model_state_dict"])

del saved


# Restore the trained Phase 1 tokenizer
with open(
    MODEL_DIR / "tokenizer.json",
    encoding="utf-8",
) as f:
    state = json.load(f)

tokenizer = BytelevelBPE(
    vocab_size=state["vocab_size"]
)

tokenizer.specials = state["specials"]
tokenizer.vocab = state["vocab"]
tokenizer.vocab_size = len(tokenizer.vocab)

tokenizer.stoi = {
    symbol: i
    for i, symbol in enumerate(tokenizer.vocab)
}

tokenizer.itos = {
    i: symbol
    for symbol, i in tokenizer.stoi.items()
}

tokenizer.merges = [
    tuple(pair)
    for pair in state["merges"]
]

tokenizer.merge_ranks = {
    tuple(record["pair"]): (
        record["rank"],
        record["merged_symbol"],
    )
    for record in state["merge_ranks"]
}

tokenizer.symbol_to_bytes = {
    symbol: bytes(values)
    for symbol, values in state["symbol_to_bytes"].items()
}

del state

PAD_ID = tokenizer.stoi["<pad>"]
UNK_ID = tokenizer.stoi["<unk>"]
BOS_ID = tokenizer.stoi["<s>"]
EOS_ID = tokenizer.stoi["</s>"]

CHUNK_LENGTH = min(
    CHUNK_LENGTH,
    cfg.max_seq_len,
)

In [8]:
DATA_DIR = Path("/kaggle/working/capstone_data")

for filename in [
    "Phase_4_CJPE/Train/train.json",
    "Phase_4_CJPE/Train/dev.json",
    "Phase_4_CJPE/Test/test.json",
]:
    hf_hub_download(
        repo_id="Exploration-Lab/CS-781-Capstone",
        repo_type="dataset",
        filename=filename,
        local_dir=str(DATA_DIR),
        token=globals().get("HF_TOKEN"),
    )

CJPE_DIR = DATA_DIR / "Phase_4_CJPE"


def read_cases(path, labeled=True):
    with open(path, encoding="utf-8") as f:
        records = json.load(f)

    cases = []
    seen = {}

    for record in records:
        case = {
            "id": record["id"],
            "text": record["text"],
            "label": (
                int(record.get("label", record.get("labels")))
                if labeled else -100
            ),
        }

        if labeled:
            key = str(case["id"])
            value = (case["text"], case["label"])

            if key in seen:
                if seen[key] != value:
                    raise ValueError(f"Conflicting case ID: {key}")
                continue

            seen[key] = value

        cases.append(case)

    return cases


train_cases = read_cases(CJPE_DIR / "Train/train.json")
dev_cases = read_cases(CJPE_DIR / "Train/dev.json")

print("Training cases:", len(train_cases))
print("Validation cases:", len(dev_cases))

Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

Training cases: 32305
Validation cases: 994


In [9]:
class CaseDataset(Dataset):
    def __init__(self, cases):
        self.cases = cases
        self.chunks = []

        capacity = CHUNK_LENGTH - 2

        for i, case in enumerate(cases):
            ids = tokenizer.encode(case["text"]) or [UNK_ID]
            starts = list(range(0, len(ids), capacity))

            if len(starts) > MAX_CHUNKS:
                selected = np.linspace(
                    0,
                    len(starts) - 1,
                    MAX_CHUNKS,
                    dtype=int,
                )
                starts = [starts[j] for j in selected]

            chunks = [
                np.array(
                    [BOS_ID]
                    + ids[start:start + capacity]
                    + [EOS_ID],
                    dtype=np.int32,
                )
                for start in starts
            ]

            self.chunks.append(chunks)

            if (i + 1) % 1000 == 0:
                print(f"Tokenized {i + 1}/{len(cases)}")

    def __len__(self):
        return len(self.cases)

    def __getitem__(self, idx):
        return {
            "id": self.cases[idx]["id"],
            "label": self.cases[idx]["label"],
            "chunks": [
                torch.tensor(chunk, dtype=torch.long)
                for chunk in self.chunks[idx]
            ],
        }


def collate_cases(examples):
    chunks = [
        chunk
        for example in examples
        for chunk in example["chunks"]
    ]

    return {
        "input_ids": pad_sequence(
            chunks,
            batch_first=True,
            padding_value=PAD_ID,
        ),
        "lengths": torch.tensor(
            [len(chunk) for chunk in chunks]
        ),
        "chunk_counts": torch.tensor(
            [len(example["chunks"]) for example in examples]
        ),
        "labels": torch.tensor(
            [example["label"] for example in examples]
        ),
        "ids": [example["id"] for example in examples],
    }


def make_loader(dataset, shuffle=False):
    return DataLoader(
        dataset,
        batch_size=DOC_BATCH_SIZE,
        shuffle=shuffle,
        collate_fn=collate_cases,
    )


def move_batch(batch):
    for key in ["input_ids", "lengths", "labels"]:
        batch[key] = batch[key].to(device)

    return batch


train_dataset = CaseDataset(train_cases)
dev_dataset = CaseDataset(dev_cases)

train_loader = make_loader(train_dataset, shuffle=True)
dev_loader = make_loader(dev_dataset)

Tokenized 1000/32305
Tokenized 2000/32305
Tokenized 3000/32305
Tokenized 4000/32305
Tokenized 5000/32305
Tokenized 6000/32305
Tokenized 7000/32305
Tokenized 8000/32305
Tokenized 9000/32305
Tokenized 10000/32305
Tokenized 11000/32305
Tokenized 12000/32305
Tokenized 13000/32305
Tokenized 14000/32305
Tokenized 15000/32305
Tokenized 16000/32305
Tokenized 17000/32305
Tokenized 18000/32305
Tokenized 19000/32305
Tokenized 20000/32305
Tokenized 21000/32305
Tokenized 22000/32305
Tokenized 23000/32305
Tokenized 24000/32305
Tokenized 25000/32305
Tokenized 26000/32305
Tokenized 27000/32305
Tokenized 28000/32305
Tokenized 29000/32305
Tokenized 30000/32305
Tokenized 31000/32305
Tokenized 32000/32305


In [10]:
class CJPEModel(nn.Module):
    def __init__(self, base):
        super().__init__()

        self.tok_emb = base.tok_emb
        self.blocks = base.blocks
        self.ln_f = base.ln_f

        self.frozen_count = len(self.blocks) - UNFREEZE_LAST_N

        self.chunk_batch_size = CHUNK_BATCH_SIZE
        self.use_checkpointing = USE_ACTIVATION_CHECKPOINTING

        self.tok_emb.requires_grad_(False)
        self.blocks.requires_grad_(False)

        for block in self.blocks[self.frozen_count:]:
            block.requires_grad_(True)

        self.ln_f.requires_grad_(True)

        self.gru = nn.GRU(
            input_size=base.cfg.d_model,
            hidden_size=GRU_HIDDEN,
            batch_first=True,
            bidirectional=True,
        )

        self.attention = nn.Sequential(
            nn.Linear(2 * GRU_HIDDEN, 128),
            nn.Tanh(),
            nn.Linear(128, 1, bias=False),
        )

        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(2 * GRU_HIDDEN, 2)

    def train(self, mode=True):
        super().train(mode)

        for block in self.blocks[:self.frozen_count]:
            block.eval()

        return self

    def encode_chunks(self, input_ids, lengths):
        x = self.tok_emb(input_ids)

        for i, block in enumerate(self.blocks):
            if (
                self.training
                and self.use_checkpointing
                and i >= self.frozen_count
            ):
                x = checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)

        x = self.ln_f(x).float()

        positions = torch.arange(
            x.size(1), device=x.device
        )[None, :]

        # Mean of text tokens only.
        mask = (
            (positions >= 1)
            & (positions < lengths[:, None] - 1)
        ).unsqueeze(-1).float()

        return (
            (x * mask).sum(dim=1)
            / mask.sum(dim=1).clamp_min(1)
        )

    def forward(self, input_ids, lengths, chunk_counts):
        vectors = []

        for start in range(
            0, len(input_ids), self.chunk_batch_size
        ):
            end = start + self.chunk_batch_size

            vectors.append(
                self.encode_chunks(
                    input_ids[start:end],
                    lengths[start:end],
                )
            )

        vectors = torch.cat(vectors)

        # Group chunks back into their documents.
        sequences = torch.split(
            vectors,
            chunk_counts.tolist(),
        )

        x = pad_sequence(sequences, batch_first=True)

        with torch.autocast(
            device_type=x.device.type,
            enabled=False,
        ):
            x = self.dropout(x.float())

            packed = pack_padded_sequence(
                x,
                chunk_counts,
                batch_first=True,
                enforce_sorted=False,
            )

            packed, _ = self.gru(packed)

            x, _ = pad_packed_sequence(
                packed,
                batch_first=True,
            )

            positions = torch.arange(
                x.size(1), device=x.device
            )[None, :]

            valid = (
                positions
                < chunk_counts.to(x.device)[:, None]
            )

            scores = self.attention(x).squeeze(-1)
            scores = scores.masked_fill(~valid, float("-inf"))

            weights = torch.softmax(scores, dim=1)

            document_vectors = (
                x * weights.unsqueeze(-1)
            ).sum(dim=1)

            return self.classifier(
                self.dropout(document_vectors)
            )


torch.manual_seed(SEED)

cjpe_model = CJPEModel(phase1_model).to(device)
del phase1_model

In [11]:
criterion = nn.CrossEntropyLoss()

backbone_parameters = []
head_parameters = []

for name, parameter in cjpe_model.named_parameters():
    if parameter.requires_grad:
        if name.startswith(("tok_emb.", "blocks.", "ln_f.")):
            backbone_parameters.append(parameter)
        else:
            head_parameters.append(parameter)

optimizer = torch.optim.AdamW(
    [
        {
            "params": backbone_parameters,
            "lr": BACKBONE_LR,
        },
        {
            "params": head_parameters,
            "lr": HEAD_LR,
        },
    ],
    weight_decay=0.01,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp and amp_dtype == torch.float16,
)

In [12]:
def get_logits(model, batch):
    return model(
        batch["input_ids"],
        batch["lengths"],
        batch["chunk_counts"],
    ).float()


@torch.inference_mode()
def evaluate(model, loader):
    model.eval()

    gold = []
    predictions = []

    for batch in loader:
        batch = move_batch(batch)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = get_logits(model, batch)

        gold.extend(batch["labels"].cpu().tolist())
        predictions.extend(logits.argmax(-1).cpu().tolist())

    score = f1_score(
        gold,
        predictions,
        labels=[0, 1],
        average="macro",
        zero_division=0,
    )

    return score, gold, predictions

In [13]:
best_f1 = -1
bad_epochs = 0

for epoch in range(1, EPOCHS + 1):
    cjpe_model.train()

    total_loss = 0
    total_cases = 0

    for step, batch in enumerate(train_loader, start=1):
        batch = move_batch(batch)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = get_logits(cjpe_model, batch)
            loss = criterion(logits, batch["labels"])

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        nn.utils.clip_grad_norm_(
            cjpe_model.parameters(),
            max_norm=1.0,
        )

        scaler.step(optimizer)
        scaler.update()

        batch_size = len(batch["ids"])

        total_loss += loss.item() * batch_size
        total_cases += batch_size

        if step % 100 == 0:
            print(
                f"Epoch {epoch} | "
                f"Batch {step}/{len(train_loader)} | "
                f"Loss: {total_loss / total_cases:.4f}"
            )

    dev_f1, _, _ = evaluate(cjpe_model, dev_loader)

    print(
        f"Epoch {epoch} | "
        f"Train loss: {total_loss / total_cases:.4f} | "
        f"Dev macro F1: {dev_f1:.4f}"
    )

    if dev_f1 > best_f1:
        best_f1 = dev_f1
        bad_epochs = 0

        torch.save(
            cjpe_model.state_dict(),
            OUTPUT_DIR / "best_f1.pt",
        )

        print("Saved best model.")

    else:
        bad_epochs += 1

        if bad_epochs >= PATIENCE:
            print("Early stopping.")
            break

Epoch 1 | Batch 100/8077 | Loss: 0.7098
Epoch 1 | Batch 200/8077 | Loss: 0.6994
Epoch 1 | Batch 300/8077 | Loss: 0.7039
Epoch 1 | Batch 400/8077 | Loss: 0.6979
Epoch 1 | Batch 500/8077 | Loss: 0.6968
Epoch 1 | Batch 600/8077 | Loss: 0.7006
Epoch 1 | Batch 700/8077 | Loss: 0.6968
Epoch 1 | Batch 800/8077 | Loss: 0.6927
Epoch 1 | Batch 900/8077 | Loss: 0.6931
Epoch 1 | Batch 1000/8077 | Loss: 0.6913
Epoch 1 | Batch 1100/8077 | Loss: 0.6926
Epoch 1 | Batch 1200/8077 | Loss: 0.6916
Epoch 1 | Batch 1300/8077 | Loss: 0.6903
Epoch 1 | Batch 1400/8077 | Loss: 0.6914
Epoch 1 | Batch 1500/8077 | Loss: 0.6908
Epoch 1 | Batch 1600/8077 | Loss: 0.6893
Epoch 1 | Batch 1700/8077 | Loss: 0.6890
Epoch 1 | Batch 1800/8077 | Loss: 0.6890
Epoch 1 | Batch 1900/8077 | Loss: 0.6884
Epoch 1 | Batch 2000/8077 | Loss: 0.6875
Epoch 1 | Batch 2100/8077 | Loss: 0.6864
Epoch 1 | Batch 2200/8077 | Loss: 0.6850
Epoch 1 | Batch 2300/8077 | Loss: 0.6848
Epoch 1 | Batch 2400/8077 | Loss: 0.6844
Epoch 1 | Batch 2500/8077

In [14]:
weights = torch.load(
    OUTPUT_DIR / "best_f1.pt",
    map_location="cpu",
    weights_only=True,
)

cjpe_model.load_state_dict(weights)
del weights

dev_f1, gold, predictions = evaluate(
    cjpe_model,
    dev_loader,
)

print("Best validation macro F1:", dev_f1)

print(
    classification_report(
        gold,
        predictions,
        labels=[0, 1],
        digits=4,
        zero_division=0,
    )
)

Best validation macro F1: 0.773026973026973
              precision    recall  f1-score   support

           0     0.7349    0.8592    0.7922       497
           1     0.8305    0.6901    0.7538       497

    accuracy                         0.7746       994
   macro avg     0.7827    0.7746    0.7730       994
weighted avg     0.7827    0.7746    0.7730       994



In [15]:
test_cases = read_cases(
    CJPE_DIR / "Test/test.json",
    labeled=False,
)

test_dataset = CaseDataset(test_cases)
test_loader = make_loader(test_dataset)


@torch.inference_mode()
def predict(model, loader):
    model.eval()
    results = []

    for batch in loader:
        batch = move_batch(batch)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = get_logits(model, batch)

        predictions = logits.argmax(-1).cpu().tolist()

        for case_id, prediction in zip(
            batch["ids"], predictions
        ):
            results.append({
                "id": case_id,
                "pred": int(prediction),
            })

    return results


test_predictions = predict(cjpe_model, test_loader)

prediction_path = OUTPUT_DIR / "prediction.json"

with open(prediction_path, "w", encoding="utf-8") as f:
    json.dump(
        test_predictions,
        f,
        ensure_ascii=False,
        indent=2,
    )

with zipfile.ZipFile(
    OUTPUT_DIR / "submission.zip",
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(prediction_path, arcname="prediction.json")

print("Saved:", OUTPUT_DIR / "submission.zip")

Tokenized 1000/1517
Saved: /kaggle/working/phase4_model1_bigru_attention/submission.zip


# **Knowledge Distillation**

In [13]:
import copy
import torch.nn.functional as F

# TEACHER_PATH = OUTPUT_DIR / "best_f1.pt"
TEACHER_PATH = Path("/kaggle/input/datasets/tejasvaidya137/best-pt-filess/best_f1.pt")
KD_DIR = Path("/kaggle/working/phase4_model1_kd")
KD_DIR.mkdir(parents=True, exist_ok=True)

TEMPERATURE = 2.0
KD_WEIGHT = 0.5
KD_EPOCHS = 3

teacher = cjpe_model

teacher.load_state_dict(
    torch.load(
        TEACHER_PATH,
        map_location="cpu",
        weights_only=True,
    )
)

teacher.requires_grad_(False)
teacher.eval()

if "optimizer" in globals():
    del optimizer

In [14]:
teacher_logits = {}

with torch.inference_mode():
    for batch in train_loader:
        batch = move_batch(batch)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = get_logits(teacher, batch)

        for case_id, row in zip(
            batch["ids"], logits.float().cpu()
        ):
            teacher_logits[str(case_id)] = row.clone()

In [15]:
teacher.cpu()

selected_layers = np.linspace(
    0,
    len(teacher.blocks) - 1,
    max(1, len(teacher.blocks) // 2),
    dtype=int,
).tolist()

student = copy.deepcopy(teacher)

student.blocks = nn.ModuleList([
    student.blocks[i] for i in selected_layers
])

student.requires_grad_(True)
student.tok_emb.requires_grad_(False)
student.frozen_count = 0

student = student.to(device)

In [16]:
backbone_params = []
head_params = []

for name, parameter in student.named_parameters():
    if parameter.requires_grad:
        if name.startswith(("blocks.", "ln_f.")):
            backbone_params.append(parameter)
        else:
            head_params.append(parameter)

kd_optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": 2e-5},
        {"params": head_params, "lr": 1e-4},
    ],
    weight_decay=0.01,
)

kd_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp and amp_dtype == torch.float16,
)

In [ ]:
best_f1 = -1.0
bad_epochs = 0

for epoch in range(1, KD_EPOCHS + 1):
    student.train()

    total_loss = 0.0
    total_cases = 0

    for step, batch in enumerate(train_loader, start=1):
        teacher_batch = torch.stack([
            teacher_logits[str(case_id)]
            for case_id in batch["ids"]
        ]).to(device)

        batch = move_batch(batch)
        kd_optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = get_logits(student, batch)

        ce_loss = F.cross_entropy(
            logits, batch["labels"]
        )

        kd_loss = F.kl_div(
            F.log_softmax(logits / TEMPERATURE, dim=-1),
            F.softmax(teacher_batch / TEMPERATURE, dim=-1),
            reduction="batchmean",
        ) * TEMPERATURE ** 2

        loss = (
            (1 - KD_WEIGHT) * ce_loss
            + KD_WEIGHT * kd_loss
        )

        kd_scaler.scale(loss).backward()
        kd_scaler.unscale_(kd_optimizer)

        nn.utils.clip_grad_norm_(student.parameters(), 1.0)

        kd_scaler.step(kd_optimizer)
        kd_scaler.update()

        batch_size = len(batch["ids"])
        total_loss += loss.item() * batch_size
        total_cases += batch_size

        if step % 100 == 0:
            print(
                f"Epoch {epoch} | "
                f"Batch {step}/{len(train_loader)} | "
                f"Loss: {total_loss / total_cases:.4f}"
            )

    dev_f1, _, _ = evaluate(student, dev_loader)

    print(
        f"Epoch {epoch} | "
        f"Loss: {total_loss / total_cases:.4f} | "
        f"Dev macro F1: {dev_f1:.4f}"
    )

    if dev_f1 > best_f1:
        best_f1 = dev_f1
        bad_epochs = 0

        torch.save(
            student.state_dict(),
            KD_DIR / "best_student.pt",
        )
    else:
        bad_epochs += 1

    if bad_epochs >= 2:
        break

In [18]:
student.load_state_dict(
    torch.load(
        KD_DIR / "best_student.pt",
        map_location="cpu",
        weights_only=True,
    )
)

dev_f1, gold, predictions = evaluate(student, dev_loader)

print(f"Student dev macro F1: {dev_f1:.4f}")
print(
    classification_report(
        gold,
        predictions,
        labels=[0, 1],
        digits=4,
        zero_division=0,
    )
)

Student dev macro F1: 0.7799
              precision    recall  f1-score   support

           0     0.7389    0.8712    0.7996       497
           1     0.8431    0.6922    0.7602       497

    accuracy                         0.7817       994
   macro avg     0.7910    0.7817    0.7799       994
weighted avg     0.7910    0.7817    0.7799       994



In [19]:
test_cases = read_cases(
    CJPE_DIR / "Test/test.json",
    labeled=False,
)

test_dataset = CaseDataset(test_cases)
test_loader = make_loader(test_dataset)


@torch.inference_mode()
def predict(model, loader):
    model.eval()
    results = []

    for batch in loader:
        batch = move_batch(batch)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = get_logits(model, batch)

        predictions = logits.argmax(-1).cpu().tolist()

        for case_id, prediction in zip(
            batch["ids"], predictions
        ):
            results.append({
                "id": case_id,
                "pred": int(prediction),
            })

    return results


test_predictions = predict(student, test_loader)

prediction_path = OUTPUT_DIR / "prediction.json"

with open(prediction_path, "w", encoding="utf-8") as f:
    json.dump(
        test_predictions,
        f,
        ensure_ascii=False,
        indent=2,
    )

with zipfile.ZipFile(
    OUTPUT_DIR / "submission.zip",
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(prediction_path, arcname="prediction.json")

print("Saved:", OUTPUT_DIR / "submission.zip")

Tokenized 1000/1517
Saved: /kaggle/working/phase4_model1_bigru_attention/submission.zip


extra

In [20]:
student.load_state_dict(
    torch.load(
        KD_DIR / "best_student.pt",
        map_location="cpu",
        weights_only=True,
    )
)

student = student.to(device)
student.eval()

all_probs = []
all_labels = []

with torch.inference_mode():
    for batch in dev_loader:
        batch = move_batch(batch)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = get_logits(student, batch)

        probs = torch.softmax(logits.float(), dim=-1)[:, 1]

        all_probs.extend(probs.cpu().tolist())
        all_labels.extend(batch["labels"].cpu().tolist())

y_true = np.array(all_labels)
p_class1 = np.array(all_probs)

In [21]:
scores = []

for threshold in np.round(np.arange(0.05, 0.951, 0.01), 2):
    predictions = (p_class1 >= threshold).astype(int)

    score = f1_score(
        y_true,
        predictions,
        labels=[0, 1],
        average="macro",
        zero_division=0,
    )

    scores.append((float(threshold), float(score)))

# Prefer the threshold closest to 0.5 when scores tie.
scores.sort(key=lambda row: (-row[1], abs(row[0] - 0.5)))

BEST_THRESHOLD, best_score = scores[0]

baseline_score = f1_score(
    y_true,
    (p_class1 >= 0.5).astype(int),
    labels=[0, 1],
    average="macro",
    zero_division=0,
)

print(f"Macro F1 at 0.50: {baseline_score:.4f}")
print(f"Selected threshold: {BEST_THRESHOLD:.2f}")
print(f"Tuned dev macro F1: {best_score:.4f}")

print(
    classification_report(
        y_true,
        (p_class1 >= BEST_THRESHOLD).astype(int),
        labels=[0, 1],
        digits=4,
        zero_division=0,
    )
)

with open(KD_DIR / "threshold.json", "w") as f:
    json.dump(
        {"threshold": BEST_THRESHOLD, "dev_macro_f1": best_score},
        f,
        indent=2,
    )

Macro F1 at 0.50: 0.7799
Selected threshold: 0.38
Tuned dev macro F1: 0.7927
              precision    recall  f1-score   support

           0     0.7859    0.8048    0.7952       497
           1     0.8000    0.7807    0.7902       497

    accuracy                         0.7928       994
   macro avg     0.7929    0.7928    0.7927       994
weighted avg     0.7929    0.7928    0.7927       994



In [22]:
test_cases = read_cases(
    CJPE_DIR / "Test/test.json",
    labeled=False,
)

test_dataset = CaseDataset(test_cases)
test_loader = make_loader(test_dataset)


@torch.inference_mode()
def predict_with_threshold(model, loader, threshold):
    model.eval()
    results = []

    for batch in loader:
        batch = move_batch(batch)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = get_logits(model, batch)

        probs = torch.softmax(logits.float(), dim=-1)[:, 1]
        predictions = (probs >= threshold).long().cpu().tolist()

        for case_id, prediction in zip(
            batch["ids"], predictions
        ):
            results.append({
                "id": case_id,
                "pred": int(prediction),
            })

    return results


test_predictions = predict_with_threshold(
    student,
    test_loader,
    BEST_THRESHOLD,
)

Tokenized 1000/1517


In [23]:
prediction_path = KD_DIR / "prediction.json"

with open(prediction_path, "w", encoding="utf-8") as f:
    json.dump(
        test_predictions,
        f,
        ensure_ascii=False,
        indent=2,
    )

with zipfile.ZipFile(
    KD_DIR / "submission.zip",
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(prediction_path, arcname="prediction.json")

print("Saved:", KD_DIR / "submission.zip")

Saved: /kaggle/working/phase4_model1_kd/submission.zip
